# Chapter 06: Hand Dynamics and Control

Source orientation: printed pp. 265-316; PDF pp. 283-334. Chapter question: **How do constrained fingers control an object without fighting their own contacts?**

This notebook is an original, standalone teaching chapter. It uses the textbook for structure and mathematical orientation, but the explanations, code, and figures are created here. The chapter focus is Pfaffian constraints, d'Alembert projection, internal forces, tendon actuation, hierarchy. The key objects are Pfaffian constraints, Lagrange multipliers, d'Alembert virtual work, constrained inertia, internal force nullspaces, tendon coupling matrices, and hierarchical object/finger controllers.

Generated artifacts live under `artifacts/chapter-06` and are displayed both by code and in the static gallery. The final sanity cell checks the mathematical claims and artifact files so that the notebook remains auditable after reruns.


## Translation Guide

The chapter treats a hand-object system as constrained dynamics. Contact equations remove forbidden velocities, while internal forces live in directions that change contact loading without accelerating the object. The computational translation used here is deliberately concrete: every named object becomes an array, graph, cone, trajectory, or map whose dimensions can be checked. That keeps the notebook independent from the PDF while preserving the chapter's mathematical route.

The main objects for this chapter are Pfaffian constraints, Lagrange multipliers, d'Alembert virtual work, constrained inertia, internal force nullspaces, tendon coupling matrices, and hierarchical object/finger controllers. Read each one as a map between spaces. Ask what frame or chart supplies its coordinates, what operation changes those coordinates, and what quantity should remain invariant. The visuals in this notebook are chosen to make those questions inspectable rather than decorative.

The source pages are used only as orientation. Definitions and examples are paraphrased into course language, and all figures are generated from fresh code. When a visual resembles a textbook concept, it is a redraw or computational experiment rather than a page crop.


## Route Through The Chapter

We move in four passes. First, the notebook names the chapter's geometric objects and their engineering purpose. Second, it generates the visual sequence: pfaffian constraint types, internal force nullspace, tendon actuation feasibility. Third, it runs a compact computational check that records the relevant residuals, ranks, endpoint errors, determinants, or signs. Fourth, it closes with an applied lab that asks the reader to change a parameter and explain what stayed invariant.

The important habit is to compare the visual artifact with the JSON check. If a cone claims feasibility, the feasibility check should agree. If a Jacobian claims usable motion, the task-space determinant or rank should agree. If a loop claims bracket displacement, the endpoint check should reveal it. The notebook is designed so those cross-checks are near each other.


In [ ]:
from pathlib import Path
import sys
import numpy as np

BOOK_ROOT = Path.cwd()
for candidate in [BOOK_ROOT, *BOOK_ROOT.parents]:
    if (candidate / "00-book-index.ipynb").exists() and (candidate / "utils").exists():
        BOOK_ROOT = candidate
        break
else:
    raise RuntimeError("Could not find the robotics course root")

if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

ARTIFACT_ROOT = BOOK_ROOT / "artifacts"
TOPIC = "chapter-06"

from utils.artifacts import display_artifact, save_json
from utils.visuals import build_storyboard, storyboard_check_payload


In [ ]:
storyboard = {
  "label": "Chapter 06: Hand Dynamics and Control",
  "artifact_topic": "chapter-06",
  "source_span": "printed pp. 265-316; PDF pp. 283-334",
  "visual_sequence": [
    {
      "kind": "constraint",
      "concept": "Pfaffian Constraint Types",
      "filename": "pfaffian-constraint-types.png",
      "observation": "velocity planes and forbidden directions"
    },
    {
      "kind": "internal",
      "concept": "Internal Force Nullspace",
      "filename": "internal-force-nullspace.png",
      "observation": "squeeze forces invisible to object motion"
    },
    {
      "kind": "tendon",
      "concept": "Tendon Actuation Feasibility",
      "filename": "tendon-actuation-feasibility.png",
      "observation": "pull-only tendons and torque cones"
    }
  ]
}

visual_results = build_storyboard(storyboard, ARTIFACT_ROOT, TOPIC)
visual_payload = storyboard_check_payload(storyboard, visual_results)
for item in visual_results:
    display_artifact(item["path"], width=720)
visual_payload


## Static Artifact Gallery

![Pfaffian Constraint Types](../../artifacts/chapter-06/figures/pfaffian-constraint-types.png)

*Inspection target:* velocity planes and forbidden directions.

![Internal Force Nullspace](../../artifacts/chapter-06/figures/internal-force-nullspace.png)

*Inspection target:* squeeze forces invisible to object motion.

![Tendon Actuation Feasibility](../../artifacts/chapter-06/figures/tendon-actuation-feasibility.png)

*Inspection target:* pull-only tendons and torque cones.


## Worked Interpretation

The executable check builds a toy constraint matrix, verifies the constrained mass matrix has full rank, separates an internal squeeze from object wrench, and tests a pull-only tendon vector through a coupling matrix. This is the chapter's small worked example, not a full simulator. It is intentionally narrow enough that the relevant convention can be seen directly in code and broad enough to catch the common failure mode.

The visual sequence and the executable check use compatible parameters whenever possible. The point is to avoid a split course where the images tell one story and the numbers tell another. When extending this notebook, reuse that pattern: pick one invariant, draw the geometry that exposes it, then save a check payload that would fail if the convention were reversed or the rank condition were lost.

**Pitfall to watch:** Substituting nonholonomic constraints too early can create the wrong equations. Keep velocity constraints, virtual displacements, and multiplier forces in their proper roles before projecting the dynamics. This pitfall is why the notebook avoids silent coordinate conventions. Names, frames, dimensions, and signs are part of the teaching surface, not implementation trivia.


In [ ]:
from scipy.linalg import null_space

A = np.array([[1.0, 0.0, -0.35, 0.0], [0.0, 1.0, 0.0, -0.25]])
M = np.diag([1.2, 1.0, 0.7, 0.6])
constraint_metric = A @ np.linalg.inv(M) @ A.T
G_object = np.array([[1.0, -1.0], [0.0, 0.0], [0.0, 0.0]])
internal = np.array([1.0, 1.0])
tendon_coupling = np.array([[1.0, 0.35, -0.15], [0.15, 0.75, 0.45]])
tensions = np.array([1.0, 0.7, 0.6])
joint_torque = tendon_coupling @ tensions
check_payload = {
    "constraint_rank": int(np.linalg.matrix_rank(A)),
    "constraint_metric_min_eigenvalue": float(np.linalg.eigvalsh(constraint_metric).min()),
    "internal_force_residual": float(np.linalg.norm(G_object @ internal)),
    "internal_force_dimension": int(null_space(G_object).shape[1]),
    "tensions_positive": bool(np.all(tensions > 0)),
    "joint_torque": joint_torque.round(6).tolist(),
}
assert check_payload["constraint_rank"] == 2
assert check_payload["constraint_metric_min_eigenvalue"] > 0
assert check_payload["internal_force_residual"] < 1e-12
assert check_payload["tensions_positive"]
check_payload


## Applied Lab

Lab prompt: Separate object wrench from internal squeeze and check tendon tension feasibility.

Before running the lab, make a prediction in three parts: which geometric object is being changed, which representation will show the change most clearly, and which scalar check should move in the same direction. After running it, compare the prediction with the saved JSON payload under `artifacts/chapter-06/labs`.

Use the pitfall above as a diagnostic. If the visual and scalar check disagree, inspect the frame convention, normalization, rank threshold, sign convention, or parameter range. The best result is not merely a green check; it is a short explanation of why the check protects the chapter's main idea.


In [ ]:
lab_notes = {
    "lab": 'Separate object wrench from internal squeeze and check tendon tension feasibility.',
    "source_orientation": "printed pp. 265-316; PDF pp. 283-334",
    "artifact_topic": TOPIC,
    "reader_task": "Change one parameter, rerun the visual cell, and explain which invariant did or did not change.",
}
lab_path = save_json(lab_notes, TOPIC, "labs", "applied-lab.json", root=ARTIFACT_ROOT)
display_artifact(lab_path)


In [ ]:
from pathlib import Path

visual_paths = [Path(item["path"]) for item in visual_results]
assert visual_payload["assertions"]["has_multiple_visuals"]
assert visual_payload["assertions"]["all_visuals_nonblank"]
assert all(path.exists() and path.stat().st_size > 200 for path in visual_paths)

final_sanity = {
    "visual_payload": visual_payload,
    "checks": check_payload,
    "visual_artifact_count": len(visual_paths),
    "visual_artifacts": [path.relative_to(BOOK_ROOT).as_posix() for path in visual_paths],
}
sanity_path = save_json(final_sanity, TOPIC, "checks", "final-sanity.json", root=ARTIFACT_ROOT)
display_artifact(sanity_path)
final_sanity


## Practice And Inspection Notes

Use this chapter as a small laboratory, not as a static summary. The source span is printed pp. 265-316 and PDF pp. 283-334, but the working material is the notebook itself: the generated artifacts, the executable check payload, and the applied lab. Start by reading the chapter question again: **How do constrained fingers control an object without fighting their own contacts?** Then identify which part of the visual sequence gives the most direct answer. For this chapter the focus is Pfaffian constraints, d'Alembert projection, internal forces, tendon actuation, hierarchy, so the useful inspection targets are not generic screenshots; they are the ranks, cones, motions, frames, phase loops, energy shapes, or dependency arrows that make that focus concrete.

The `pfaffian constraint types` artifact uses the `constraint` visual family; inspect it for velocity planes and forbidden directions. The `internal force nullspace` artifact uses the `internal` visual family; inspect it for squeeze forces invisible to object motion. The `tendon actuation feasibility` artifact uses the `tendon` visual family; inspect it for pull-only tendons and torque cones.

After viewing the artifacts, rerun the computational check cell and read the keys in `check_payload` as a miniature rubric. Each key should protect one concept from a plausible robotics mistake. A determinant or rank protects a singularity claim. A residual protects an equation or closure condition. A monotonicity flag protects a scale-law interpretation. An endpoint error protects a steering construction. A power-invariance error protects a frame transformation. If a check is near zero, ask why zero is the right target; if a check is positive, ask what physical or geometric margin it measures.

Try three variations. First, make a small parameter change that should preserve the chapter's main invariant, such as a coordinate-frame change, a harmless redraw scale, or a sample density increase. Second, make a parameter change that should stress the model, such as a near-singular joint pose, lower friction coefficient, larger controller delay, smaller bracket loop, or weaker tendon tension. Third, make a convention change deliberately, such as reversing a normal or swapping a body/spatial interpretation, and predict which check should fail. These variations turn the notebook from a solved example into a diagnostic tool.

When writing your own notes, use this sentence pattern: "The artifact shows ..., the computation checks ..., and the invariant is ... ." That pattern is intentionally repetitive because it catches vague understanding. A reader who can fill in those three blanks for this chapter can usually transfer the idea to a different mechanism, contact model, or control task without reopening the textbook.


## Takeaways

- Hand control is constrained robot dynamics, not just several independent finger controllers.
- Internal forces are useful precisely because they lie in the grasp map nullspace.
- Tendon actuation adds another cone: feasible joint torques must come from nonnegative tensions.
- Revisit the saved artifacts after changing parameters; the strongest learning usually comes from explaining why the invariant survived.
